# BulkFormer DX Demo Pipeline (37M)

This notebook reproduces the full demo pipeline: preprocess, spike inject, anomaly score, calibrate, benchmark metrics, and report generation.

Run cells top-to-bottom. Each cell executes CLI commands via `subprocess`. Requires:
- `data/demo_count_data.csv`, `data/gene_length_df.csv`
- BulkFormer assets and `model/BulkFormer_37M.pt`

In [1]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path('..').resolve()
env = os.environ.copy()
env['PYTHONPATH'] = str(ROOT)

def run(cmd: list[str], cwd: Path = ROOT) -> int:
    """Run command with PYTHONPATH=ROOT, return exit code."""
    print(' '.join(cmd))
    r = subprocess.run(cmd, cwd=cwd, env=env)
    return r.returncode

## 1. Preprocess

In [2]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'preprocess',
     '--counts', 'data/demo_count_data.csv',
     '--annotation', 'data/gene_length_df.csv',
     '--output-dir', 'runs/demo_preprocess_37M',
     '--counts-orientation', 'samples-by-genes',
     '--expression-filter', 'outrider_like'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli preprocess --counts data/demo_count_data.csv --annotation data/gene_length_df.csv --output-dir runs/demo_preprocess_37M --counts-orientation samples-by-genes
Wrote preprocessing outputs to runs/demo_preprocess_37M
Samples: 100 | input genes: 20010 | BulkFormer-valid genes: 20010/20010


0

## 2. Spike inject

In [3]:
run([sys.executable, 'scripts/demo_spike_inject.py'],
    cwd=ROOT)  # PYTHONPATH=. implied when run from repo root

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/demo_spike_inject.py
Injected 40 outliers. Output: /home/smirnov/projects/BulkFormer/runs/demo_spike_37M/aligned_log1p_tpm_spiked.tsv


0

## 3. Anomaly score (base)

In [4]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'score',
     '--input', 'runs/demo_preprocess_37M/aligned_log1p_tpm.tsv',
     '--valid-gene-mask', 'runs/demo_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/demo_anomaly_score_37M',
     '--variant', '37M', '--device', 'cuda', '--mask-schedule', 'deterministic', '--K-target', '5', '--mask-prob', '0.10'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly score --input runs/demo_preprocess_37M/aligned_log1p_tpm.tsv --valid-gene-mask runs/demo_preprocess_37M/valid_gene_mask.tsv --output-dir runs/demo_anomaly_score_37M --variant 37M --device cuda --mc-passes 16 --mask-prob 0.15


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote anomaly scoring outputs to runs/demo_anomaly_score_37M
Samples: 100 | valid genes: 20010 | MC passes: 16 | mean cohort abs residual: 0.7681


0

## 4. Calibrate (base)

In [5]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'calibrate',
     '--scores', 'runs/demo_anomaly_score_37M',
     '--output-dir', 'runs/demo_anomaly_calibrated_37M',
     '--alpha', '0.05'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores runs/demo_anomaly_score_37M --output-dir runs/demo_anomaly_calibrated_37M --alpha 0.05
Wrote calibrated anomaly outputs to runs/demo_anomaly_calibrated_37M
Samples: 100 | scored genes: 1852438 | method: none
Outlier counts: mean=438.5 | median=306 | max=3558 | inflation=0.47x


0

## 5. Anomaly score (spiked)

In [6]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'score',
     '--input', 'runs/demo_spike_37M/aligned_log1p_tpm_spiked.tsv',
     '--valid-gene-mask', 'runs/demo_preprocess_37M/valid_gene_mask.tsv',
     '--output-dir', 'runs/demo_spike_anomaly_score_37M',
     '--variant', '37M', '--device', 'cuda', '--mask-schedule', 'deterministic', '--K-target', '5', '--mask-prob', '0.10'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly score --input runs/demo_spike_37M/aligned_log1p_tpm_spiked.tsv --valid-gene-mask runs/demo_preprocess_37M/valid_gene_mask.tsv --output-dir runs/demo_spike_anomaly_score_37M --variant 37M --device cuda --mc-passes 16 --mask-prob 0.15


/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:68: UserWarning: An issue occurred while importing 'pyg-lib'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/libpyg.so
  warnings.warn(f"An issue occurred while importing 'pyg-lib'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:86: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_scatter/_version_cuda.so
  warnings.warn(f"An issue occurred while importing 'torch-scatter'. "
/srv/miniconda3/envs/bulkformer-cuda/lib/python3.12/site-packages/torch_geometric/typing.py:97: UserWarning: An issue occurred while importing 'torch-cluster'. Disabling its usage. Stacktrace: Could not load this library: /srv/miniconda3/envs/bulkformer

Wrote anomaly scoring outputs to runs/demo_spike_anomaly_score_37M
Samples: 100 | valid genes: 20010 | MC passes: 16 | mean cohort abs residual: 0.7682


0

## 6. Calibrate (spiked)

In [7]:
run([sys.executable, '-m', 'bulkformer_dx.cli', 'anomaly', 'calibrate',
     '--scores', 'runs/demo_spike_anomaly_score_37M',
     '--output-dir', 'runs/demo_spike_anomaly_calibrated_37M',
     '--alpha', '0.05'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python -m bulkformer_dx.cli anomaly calibrate --scores runs/demo_spike_anomaly_score_37M --output-dir runs/demo_spike_anomaly_calibrated_37M --alpha 0.05
Wrote calibrated anomaly outputs to runs/demo_spike_anomaly_calibrated_37M
Samples: 100 | scored genes: 1852438 | method: none
Outlier counts: mean=438.5 | median=306 | max=3562 | inflation=0.47x


0

## 7. Spike recovery metrics

In [8]:
run([sys.executable, 'scripts/spike_recovery_metrics.py'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/spike_recovery_metrics.py


/home/smirnov/projects/BulkFormer/bulkformer_dx/benchmark/plots.py:217: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.tight_layout()
/home/smirnov/projects/BulkFormer/bulkformer_dx/benchmark/plots.py:218: UserWarning: Creating legend with loc="best" can be slow with large amounts of data.
  fig.savefig(output_path, dpi=150)


Spike recovery: /home/smirnov/projects/BulkFormer/reports/spike_recovery.tsv
Benchmark metrics: AUROC=0.7104 AUPRC=0.0001


0

## 8. Generate report

In [9]:
run([sys.executable, 'scripts/generate_demo_report.py'])

/srv/miniconda3/envs/bulkformer-cuda/bin/python scripts/generate_demo_report.py
Wrote /home/smirnov/projects/BulkFormer/reports/bulkformer_dx_demo_report.md


0

## Summary

In [10]:
import json
qc = json.loads((ROOT / 'reports' / 'anomaly_qc_summary.json').read_text())
print('Benchmark metrics:')
for k in ['auroc', 'auprc', 'recall_at_fdr_05', 'spike_targets_significant_after']:
    if k in qc:
        print(f'  {k}: {qc[k]}')

Benchmark metrics:
  auroc: 0.7103971281284983
  auprc: 0.0001014774427941189
  recall_at_fdr_05: 0.375
  spike_targets_significant_after: 15
